In [ ]:
# ==============================================
# SETUP: Run this cell first to get latest code
# ==============================================
# NOTEBOOK VERSION: 2024-01-30-v8 (outlines for structured output)
# ==============================================
import os

NOTEBOOK_VERSION = "2024-01-30-v8"

# Check if we're already in the repo
if os.path.exists('mycelium'):
    print("Repo exists, pulling latest...")
    %cd mycelium
    !git pull
elif os.path.basename(os.getcwd()) == 'mycelium':
    print("Already in mycelium, pulling latest...")
    !git pull
else:
    print("Cloning repo...")
    !git clone https://github.com/bryceroche/mycelium.git
    %cd mycelium

# Show version info
print(f"\n{'='*50}")
print(f"NOTEBOOK VERSION: {NOTEBOOK_VERSION}")
!git log -1 --format="GIT COMMIT: %h %s"
print(f"{'='*50}")
print(f"Current directory: {os.getcwd()}")
print("Setup complete!")

In [ ]:
# Install dependencies - ROBUST COLAB VERSION
# NOTEBOOK VERSION: 2024-01-30-v8 (outlines for structured output)
import sys
print(f"Python: {sys.executable}")

# Ensure we're installing to the RIGHT Python environment
print("Step 1: Clear pip cache...")
!pip cache purge

print("\nStep 2: Uninstall old transformers...")
!{sys.executable} -m pip uninstall transformers -y

print("\nStep 3: Install transformers >=4.40 with no cache...")
!{sys.executable} -m pip install transformers>=4.40.0 --no-cache-dir

print("\nStep 4: Install other deps...")
!{sys.executable} -m pip install accelerate sentencepiece datasets bitsandbytes --quiet

print("\nStep 5: Install outlines for structured generation...")
!{sys.executable} -m pip install outlines --quiet

# Now import fresh
print("\nStep 6: Import and verify...")
import transformers
import outlines
print(f"Transformers version: {transformers.__version__}")
print(f"Outlines version: {outlines.__version__}")

if tuple(map(int, transformers.__version__.split('.')[:2])) < (4, 40):
    print("\n" + "!"*60)
    print("WARNING: transformers < 4.40 detected!")
    print("RESTART RUNTIME and run this cell again.")
    print("!"*60)
else:
    print("✓ All dependencies ready!")

In [ ]:
# Core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from dataclasses import dataclass
import json
import math
from typing import Optional, Dict, List, Tuple

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load GSM8K directly from Hugging Face (no manual upload needed!)
from datasets import load_dataset
import re

print("Loading GSM8K from Hugging Face...")
gsm8k = load_dataset("gsm8k", "main")

def extract_answer(answer_str: str) -> float:
    """Extract numeric answer from GSM8K answer string."""
    # GSM8K format: reasoning text followed by #### <number>
    match = re.search(r'####\s*([0-9,.-]+)', answer_str)
    if match:
        return float(match.group(1).replace(',', ''))
    # Fallback: last number in string
    numbers = re.findall(r'[0-9,]+\.?[0-9]*', answer_str)
    if numbers:
        return float(numbers[-1].replace(',', ''))
    return 0.0

def convert_to_our_format(example):
    """Convert HF format to our format."""
    return {
        "question": example["question"],
        "normalized_question": example["question"],  # Same for now
        "answer": extract_answer(example["answer"]),
        "raw_answer": example["answer"],
    }

# Convert datasets
train_data = [convert_to_our_format(ex) for ex in gsm8k["train"]]
test_data = [convert_to_our_format(ex) for ex in gsm8k["test"]]

print(f"Loaded {len(train_data)} train, {len(test_data)} test problems")
print(f"\nExample:")
print(f"  Q: {train_data[0]['question'][:80]}...")
print(f"  A: {train_data[0]['answer']}")

## 1. Signature Storage (The Tree)

In [ ]:
@dataclass
class Signature:
    """A single signature (leaf node) in the tree."""
    id: int
    dsl: str  # The operation code, e.g., "a + b"
    description: str  # Natural language description
    
    # Welford online stats
    n: int = 0
    mean: float = 0.5  # Success rate
    m2: float = 0.0    # For variance calculation
    
    @property
    def variance(self) -> float:
        if self.n < 2:
            return 1.0  # High variance = uncertain
        return self.m2 / (self.n - 1)
    
    def update(self, success: bool):
        """Welford online update."""
        x = 1.0 if success else 0.0
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.m2 += delta * delta2


class SignatureTree:
    """Growable tree of signatures with deduplication."""
    
    def __init__(self, embedding_dim: int = 256, max_signatures: int = 500):
        self.embedding_dim = embedding_dim
        self.max_signatures = max_signatures
        
        # Signature metadata
        self.signatures: Dict[int, Signature] = {}
        
        # DSL to ID mapping for deduplication
        self.dsl_to_id: Dict[str, int] = {}
        
        # Embedding storage (will be moved to GPU)
        self.embeddings = torch.zeros(max_signatures, embedding_dim)
        self.active_mask = torch.zeros(max_signatures, dtype=torch.bool)
        self.num_active = 0
        
        # Special signatures
        self.DONE_IDX = 0  # Reserved for "done" signal
        self._init_special_signatures()
    
    def _init_special_signatures(self):
        """Initialize special signatures."""
        # DONE signature
        self.signatures[0] = Signature(
            id=0,
            dsl="DONE",
            description="Problem solved, return final answer"
        )
        self.dsl_to_id["DONE"] = 0
        self.embeddings[0] = torch.randn(self.embedding_dim) * 0.1
        self.active_mask[0] = True
        self.num_active = 1
    
    def to(self, device):
        """Move to device."""
        self.embeddings = self.embeddings.to(device)
        self.active_mask = self.active_mask.to(device)
        return self
    
    @property
    def device(self):
        """Get current device of embeddings."""
        return self.embeddings.device
    
    def has_dsl(self, dsl: str) -> bool:
        """Check if DSL already exists."""
        return dsl in self.dsl_to_id
    
    def get_id_for_dsl(self, dsl: str) -> Optional[int]:
        """Get signature ID for a DSL if it exists."""
        return self.dsl_to_id.get(dsl)
    
    def add_signature(self, embedding: torch.Tensor, dsl: str, description: str) -> Tuple[int, bool]:
        """
        Add a new signature to the tree.
        
        Returns: (signature_id, was_created)
        - If DSL exists, returns existing ID and False
        - If DSL is new, creates signature and returns new ID and True
        """
        # Detach and move to correct device - no gradients for tree storage
        with torch.no_grad():
            embedding = embedding.detach().to(self.device)
            
            # Check for duplicate DSL
            if dsl in self.dsl_to_id:
                existing_id = self.dsl_to_id[dsl]
                # Update embedding with running average (helps clustering)
                alpha = 0.1
                self.embeddings[existing_id] = (1 - alpha) * self.embeddings[existing_id] + alpha * embedding
                return existing_id, False
            
            if self.num_active >= self.max_signatures:
                # At capacity - find most similar existing signature
                return self._find_most_similar(embedding), False
            
            idx = self.num_active
            
            self.signatures[idx] = Signature(
                id=idx,
                dsl=dsl,
                description=description
            )
            
            self.dsl_to_id[dsl] = idx
            self.embeddings[idx] = embedding.clone()
            self.active_mask[idx] = True
            self.num_active += 1
        
        return idx, True
    
    def _find_most_similar(self, embedding: torch.Tensor) -> int:
        """Find most similar existing signature."""
        active_embs = self.embeddings[:self.num_active]
        sims = torch.cosine_similarity(embedding.unsqueeze(0), active_embs)
        return sims.argmax().item()
    
    def get_active_embeddings(self) -> torch.Tensor:
        """Get embeddings of active signatures (cloned to avoid in-place issues)."""
        return self.embeddings[:self.num_active].clone()
    
    def get_variances(self) -> torch.Tensor:
        """Get Welford variances for active signatures."""
        variances = torch.tensor([
            self.signatures[i].variance 
            for i in range(self.num_active)
        ])
        return variances
    
    def update_signature(self, idx: int, success: bool):
        """Update Welford stats for a signature."""
        if idx in self.signatures:
            self.signatures[idx].update(success)
    
    def get_dsl(self, idx: int) -> str:
        """Get DSL code for a signature."""
        return self.signatures.get(idx, Signature(0, "UNKNOWN", "")).dsl
    
    def summary(self) -> str:
        """Get summary of unique DSLs and their stats."""
        lines = []
        for dsl, idx in sorted(self.dsl_to_id.items(), key=lambda x: x[1]):
            sig = self.signatures[idx]
            lines.append(f"[{idx}] {dsl:12s} n={sig.n:3d} success={sig.mean:.0%} var={sig.variance:.2f}")
        return "\n".join(lines)


# Test it
tree = SignatureTree(embedding_dim=256)
print(f"Initialized tree with {tree.num_active} signatures")

# Test deduplication
emb1 = torch.randn(256)
idx1, created1 = tree.add_signature(emb1, "a + b", "addition")
print(f"Added 'a + b': idx={idx1}, created={created1}")

emb2 = torch.randn(256)
idx2, created2 = tree.add_signature(emb2, "a + b", "addition again")
print(f"Added 'a + b' again: idx={idx2}, created={created2}")

emb3 = torch.randn(256)
idx3, created3 = tree.add_signature(emb3, "a - b", "subtraction")
print(f"Added 'a - b': idx={idx3}, created={created3}")

print(f"\nTree now has {tree.num_active} signatures (deduplicated)")

## 2. Tree-Bound LLM Model

In [ ]:
class TreeCrossAttention(nn.Module):
    """Cross-attention layer from LLM hidden states to tree signatures."""
    
    def __init__(self, hidden_size: int, signature_dim: int, num_heads: int = 4):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.signature_dim = signature_dim
        self.num_heads = num_heads
        self.head_dim = signature_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_size, signature_dim)
        self.k_proj = nn.Linear(signature_dim, signature_dim)
        self.out_proj = nn.Linear(signature_dim, signature_dim)
        self.norm = nn.LayerNorm(signature_dim)
        
    def forward(
        self, 
        hidden_states: torch.Tensor,  # [batch, hidden_size]
        signature_embeddings: torch.Tensor,  # [num_sigs, sig_dim]
        welford_variances: torch.Tensor,  # [num_sigs]
        return_weights: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Cross-attention to tree."""
        Q = self.q_proj(hidden_states)
        K = self.k_proj(signature_embeddings)
        
        scores = torch.matmul(Q, K.T) / math.sqrt(self.signature_dim)
        temperature = 1.0 + welford_variances.unsqueeze(0)
        adjusted_scores = scores / temperature
        attn_weights = F.softmax(adjusted_scores, dim=-1)
        
        return Q, attn_weights, scores


class TreeBoundLLM(nn.Module):
    """LLM with native tree binding via cross-attention."""
    
    def __init__(
        self, 
        base_model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        signature_dim: int = 256,
        freeze_llm: bool = True,
        min_signatures_before_routing: int = 5,
    ):
        super().__init__()
        
        self.min_signatures_before_routing = min_signatures_before_routing
        
        print(f"Loading {base_model_name}...")
        self.llm = AutoModel.from_pretrained(base_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        if freeze_llm:
            print("Freezing LLM parameters...")
            for param in self.llm.parameters():
                param.requires_grad = False
        
        hidden_size = self.llm.config.hidden_size
        self.signature_dim = signature_dim
        self.hidden_size = hidden_size
        
        self.tree_attention = TreeCrossAttention(
            hidden_size=hidden_size,
            signature_dim=signature_dim
        )
        
        self.base_threshold = nn.Parameter(torch.tensor(0.5))
        
        print(f"TreeBoundLLM initialized:")
        print(f"  LLM hidden size: {hidden_size}")
        print(f"  Signature dim: {signature_dim}")
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,}")
    
    def encode_two_stream(
        self,
        problem_text: str,
        prefix_steps: List[str],
        num_map: Dict[str, str],
        device
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Two-stream encoding: separate operation from context.
        
        Stream 1 (Operation): Minimal - just "add", "divide", etc.
        Stream 2 (Context): Values - "step 1: 48, 2"
        
        Returns: (op_embeddings, ctx_embeddings) each [num_steps, hidden_size]
        """
        if not prefix_steps:
            return None, None
        
        op_texts = []
        ctx_texts = []
        
        # Map operator symbols to words
        op_to_word = {
            '+': 'add',
            '-': 'subtract', 
            '*': 'multiply',
            '/': 'divide'
        }
        
        for i, prefix in enumerate(prefix_steps):
            op, left, right = parse_prefix_step(prefix)
            
            # Stream 1: Minimal operation description
            # Handle None or unknown operators
            if op is None:
                op_word = 'unknown'
            else:
                op_word = op_to_word.get(op, op)  # Fallback to symbol if unknown
            op_texts.append(op_word)
            
            # Stream 2: Context with values
            left_val = num_map.get(left, left) if num_map and left else str(left)
            right_val = num_map.get(right, right) if num_map and right else str(right)
            ctx_text = f"step {i+1}: {left_val}, {right_val}"
            ctx_texts.append(ctx_text)
        
        # Encode operation stream (batched)
        op_inputs = self.tokenizer(
            op_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        op_outputs = self.llm(**op_inputs)
        op_embeddings = op_outputs.last_hidden_state[:, -1, :].float()
        
        # Encode context stream (batched)
        ctx_inputs = self.tokenizer(
            ctx_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        ctx_outputs = self.llm(**ctx_inputs)
        ctx_embeddings = ctx_outputs.last_hidden_state[:, -1, :].float()
        
        return op_embeddings, ctx_embeddings
    
    def route_with_op_stream(
        self,
        op_embeddings: torch.Tensor,  # [num_steps, hidden_size]
        tree: 'SignatureTree',
        debug: bool = False
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Route using operation embeddings only (cleaner signal).
        
        Returns: (queries, route_probs, scores) each [num_steps, ...]
        """
        sig_embeddings = tree.get_active_embeddings().to(op_embeddings.device)
        variances = tree.get_variances().to(op_embeddings.device)
        
        queries, route_probs, scores = self.tree_attention(
            op_embeddings, sig_embeddings, variances
        )
        
        if debug:
            print(f"    route_with_op_stream: {op_embeddings.shape[0]} steps → {route_probs.shape}")
        
        return queries, route_probs, scores
    
    # Legacy methods below for backwards compatibility
    def encode_text(self, text: str, device) -> torch.Tensor:
        """Encode a single text to hidden state."""
        inputs = self.tokenizer(
            text,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = self.llm(**inputs)
        
        return outputs.last_hidden_state[:, -1, :].float()
    
    def encode_steps_batched(
        self, 
        problem_text: str, 
        prefix_steps: List[str],
        num_map: Dict[str, str],
        device
    ) -> torch.Tensor:
        """Legacy: single-stream encoding with mixed op+context."""
        if not prefix_steps:
            return None
        
        step_texts = []
        for i, prefix in enumerate(prefix_steps):
            op, left, right = parse_prefix_step(prefix)
            left_val = num_map.get(left, left) if num_map else left
            right_val = num_map.get(right, right) if num_map else right
            op_str = op if op else '?'
            step_text = f"solve: {problem_text} [step {i+1}: compute {left_val} {op_str} {right_val}]"
            step_texts.append(step_text)
        
        inputs = self.tokenizer(
            step_texts,
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        outputs = self.llm(**inputs)
        step_embeddings = outputs.last_hidden_state[:, -1, :].float()
        
        return step_embeddings
    
    def route_steps(
        self,
        step_embeddings: torch.Tensor,
        tree: 'SignatureTree',
        debug: bool = False
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Legacy: route using mixed embeddings."""
        sig_embeddings = tree.get_active_embeddings().to(step_embeddings.device)
        variances = tree.get_variances().to(step_embeddings.device)
        queries, route_probs, scores = self.tree_attention(
            step_embeddings, sig_embeddings, variances
        )
        if debug:
            print(f"    route_steps: {step_embeddings.shape[0]} steps → {route_probs.shape}")
        return queries, route_probs, scores
    
    def get_hidden_state(self, input_ids, attention_mask, debug=False) -> torch.Tensor:
        outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, -1, :].float()
    
    def forward(self, input_ids, attention_mask, tree, debug=False):
        hidden = self.get_hidden_state(input_ids, attention_mask, debug)
        sig_embeddings = tree.get_active_embeddings().to(hidden.device)
        variances = tree.get_variances().to(hidden.device)
        query, route_probs, scores = self.tree_attention(hidden, sig_embeddings, variances)
        should_create = self.should_create_signature(tree, scores, debug)
        return query, route_probs, scores, should_create
    
    def should_create_signature(self, tree, scores, debug=False) -> bool:
        num_sigs = tree.num_active
        if num_sigs < self.min_signatures_before_routing:
            import random
            return random.random() < 0.7
        max_score = scores.max().item()
        threshold = self.base_threshold.detach().item()
        maturity = min(1.0, num_sigs / 50)
        return max_score < threshold + 0.3 * maturity


print("TreeBoundLLM with two-stream encoding ready.")

In [ ]:
# Initialize model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256).to(device)

print(f"\nTree has {tree.num_active} signatures")

In [ ]:
# Test forward pass
test_text = "solve: John has 5 apples. Mary gives him 3 more. How many apples does John have?"
inputs = model.tokenizer(test_text, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    query, route_probs, scores, should_create = model(
        inputs["input_ids"], 
        inputs["attention_mask"],
        tree
    )

print(f"Query shape: {query.shape}")
print(f"Route probs shape: {route_probs.shape}")
print(f"Should create new signature: {should_create}")
print(f"Route probs: {route_probs}")

## 3. DSL Execution

In [ ]:
def parse_prefix_step(prefix: str) -> Tuple[str, str, str]:
    """
    Parse a prefix notation step into (operator, left, right).
    
    NOTE: This is a SIMPLE parser for basic binary ops only.
    For nested expressions, use parse_prefix_recursive() and execute_prefix_recursive().
    
    Examples:
        "/ NUM_0 CONST_0" → ("/", "NUM_0", "CONST_0")
        "+ NUM_0 step_1" → ("+", "NUM_0", "step_1")
    """
    parts = prefix.strip().split()
    if len(parts) >= 3:
        op = parts[0]
        return op, parts[1], parts[2]
    elif len(parts) == 2:
        return parts[0], parts[1], None
    else:
        return None, prefix, None


def parse_prefix_recursive(tokens: list, pos: int = 0) -> Tuple[any, int]:
    """
    Recursively parse prefix notation into an expression tree.
    
    Returns: (expression_tree, next_position)
    
    Expression tree is either:
    - A string (variable like "NUM_0", "step_1")
    - A tuple (op, left_tree, right_tree)
    
    Examples:
        "+ NUM_0 NUM_1" → ('+', 'NUM_0', 'NUM_1')
        "+ + a b c" → ('+', ('+', 'a', 'b'), 'c')
        "- - - a b c d" → ('-', ('-', ('-', 'a', 'b'), 'c'), 'd')
    """
    if pos >= len(tokens):
        return None, pos
    
    token = tokens[pos]
    
    # Check if it's an operator
    if token in ['+', '-', '*', '/']:
        # Parse left operand
        left, pos = parse_prefix_recursive(tokens, pos + 1)
        # Parse right operand
        right, pos = parse_prefix_recursive(tokens, pos)
        return (token, left, right), pos
    else:
        # It's a value (NUM_0, step_1, CONST_0, etc.)
        return token, pos + 1


def evaluate_tree(tree, num_map: Dict[str, str], step_results: Dict[str, float]) -> Optional[float]:
    """
    Evaluate an expression tree.
    
    Args:
        tree: Either a string (variable) or tuple (op, left, right)
        num_map: Maps NUM_0, CONST_0 etc to values
        step_results: Maps step_1, step_2 etc to computed values
    """
    if tree is None:
        return None
    
    # Base case: it's a variable
    if isinstance(tree, str):
        if tree in num_map:
            return float(num_map[tree])
        if tree in step_results:
            return step_results[tree]
        try:
            return float(tree)
        except:
            return None
    
    # Recursive case: it's an operation
    op, left, right = tree
    
    left_val = evaluate_tree(left, num_map, step_results)
    right_val = evaluate_tree(right, num_map, step_results)
    
    if left_val is None:
        return None
    if right_val is None:
        return None
    
    if op == '+': return left_val + right_val
    if op == '-': return left_val - right_val
    if op == '*': return left_val * right_val
    if op == '/': return left_val / right_val if right_val != 0 else None
    
    return None


def execute_prefix_step_recursive(prefix: str, num_map: Dict[str, str], step_results: Dict[str, float]) -> Optional[float]:
    """
    Execute a prefix notation step, handling nested expressions.
    
    Examples:
        "/ NUM_0 CONST_0" → simple division
        "+ + step_3 step_2 step_1" → (step_3 + step_2) + step_1
        "- - - NUM_0 NUM_1 NUM_2 NUM_3" → ((NUM_0 - NUM_1) - NUM_2) - NUM_3
    """
    tokens = prefix.strip().split()
    if not tokens:
        return None
    
    tree, _ = parse_prefix_recursive(tokens, 0)
    return evaluate_tree(tree, num_map, step_results)


def execute_prefix_step(prefix: str, num_map: Dict[str, str], step_results: Dict[str, float]) -> Optional[float]:
    """
    Execute a single prefix notation step - NOW WITH RECURSIVE SUPPORT.
    """
    # Use the recursive parser for all cases
    return execute_prefix_step_recursive(prefix, num_map, step_results)


def execute_multi_step(prefix_steps: List[str], num_map: Dict[str, str]) -> Tuple[Optional[float], Dict[str, float]]:
    """
    Execute multiple prefix steps in sequence, chaining results.
    
    Returns: (final_result, all_step_results)
    """
    step_results = {}
    
    for i, prefix in enumerate(prefix_steps):
        step_name = f"step_{i + 1}"
        result = execute_prefix_step(prefix, num_map, step_results)
        
        if result is None:
            return None, step_results
        
        step_results[step_name] = result
    
    final = list(step_results.values())[-1] if step_results else None
    return final, step_results


# Test the recursive parser
print("Testing RECURSIVE prefix parsing:")
print("="*60)

# Simple case
test1 = "+ NUM_0 NUM_1"
tokens1 = test1.split()
tree1, _ = parse_prefix_recursive(tokens1)
print(f"'{test1}' → {tree1}")

# Nested case
test2 = "+ + a b c"
tokens2 = test2.split()
tree2, _ = parse_prefix_recursive(tokens2)
print(f"'{test2}' → {tree2}")

# Deeply nested
test3 = "- - - NUM_0 NUM_1 NUM_2 NUM_3"
tokens3 = test3.split()
tree3, _ = parse_prefix_recursive(tokens3)
print(f"'{test3}' → {tree3}")

# The failing case from our data
test4 = "+ + step_3 step_2 step_3"
tokens4 = test4.split()
tree4, _ = parse_prefix_recursive(tokens4)
print(f"'{test4}' → {tree4}")

# Test execution
print("\n" + "="*60)
print("Testing RECURSIVE execution:")
print("="*60)

# Test case from failures: '+ + + + NUM_1 NUM_2 NUM_3 NUM_4 NUM_5'
# Expected: 10 + 15 + 27 + 12 + 19 = 83 (but answer was 166, so data might be wrong)
num_map_test = {'NUM_1': '10', 'NUM_2': '15', 'NUM_3': '27', 'NUM_4': '12', 'NUM_5': '19'}
prefix_test = "+ + + + NUM_1 NUM_2 NUM_3 NUM_4 NUM_5"
result = execute_prefix_step(prefix_test, num_map_test, {})
print(f"'{prefix_test}'")
print(f"num_map: {num_map_test}")
print(f"Result: {result}")

# Test the Nina toys example
print("\n" + "-"*60)
num_map_nina = {'NUM_0': '10', 'NUM_1': '5', 'NUM_2': '6', 'CONST_0': '3', 'CONST_1': '2'}
steps_nina = ['* CONST_0 NUM_0', '* CONST_1 NUM_1', '* NUM_1 NUM_2', '+ + step_3 step_2 step_3']
result_nina, history_nina = execute_multi_step(steps_nina, num_map_nina)
print(f"Nina toys example:")
print(f"Steps: {steps_nina}")
print(f"History: {history_nina}")
print(f"Result: {result_nina} (expected: 70)")

## 4. Signature Generation (LLM creates DSL)

In [ ]:
# LLM-based problem decomposition using Outlines for GUARANTEED structured output
# Outlines uses constrained decoding - invalid tokens are masked during generation

import outlines
from pydantic import BaseModel
from typing import List, Literal

# Define the output schema with Pydantic
class Step(BaseModel):
    left: float
    op: Literal["+", "-", "*", "/"]
    right: float
    result: float
    why: str

class Decomposition(BaseModel):
    steps: List[Step]
    answer: float


DECOMPOSE_PROMPT = """Break down this math problem into atomic steps.

RULES:
1. Each step has EXACTLY TWO numbers (no expressions)
2. Use only: +, -, *, /
3. Output valid JSON matching the schema

Example:
Problem: Tim has 10 dollars. He buys 3 toys at 2 dollars each. How much left?
{"steps": [{"left": 3, "op": "*", "right": 2, "result": 6, "why": "cost of toys"}, {"left": 10, "op": "-", "right": 6, "result": 4, "why": "money left"}], "answer": 4}

Example:
Problem: A baker made 24 cookies. She sold 8 in the morning and 6 in the afternoon. How many left?
{"steps": [{"left": 8, "op": "+", "right": 6, "result": 14, "why": "total sold"}, {"left": 24, "op": "-", "right": 14, "result": 10, "why": "remaining"}], "answer": 10}

Problem: {problem}
"""


def decompose_problem_outlines(problem: str, generator) -> dict:
    """Use Outlines for guaranteed structured decomposition."""
    prompt = DECOMPOSE_PROMPT.format(problem=problem)
    
    try:
        result = generator(prompt)
        return {
            'steps': [s.model_dump() for s in result.steps],
            'answer': result.answer,
            'raw_response': result.model_dump_json(),
        }
    except Exception as e:
        return {
            'steps': [],
            'answer': None,
            'raw_response': str(e),
        }


def decomposition_to_execution(decomposition: dict) -> Tuple[Optional[float], dict]:
    """Execute the decomposition and return result."""
    steps = decomposition.get('steps', [])
    step_results = {}
    
    for i, step in enumerate(steps):
        op = step.get('op', '')
        left_val = step.get('left')
        right_val = step.get('right')
        
        if left_val is None or right_val is None:
            return None, step_results
        
        if op == '+': result = left_val + right_val
        elif op == '-': result = left_val - right_val
        elif op == '*': result = left_val * right_val
        elif op == '/': result = left_val / right_val if right_val != 0 else None
        else: result = None
        
        if result is None:
            return None, step_results
        
        step_results[f'step_{i+1}'] = result
    
    final = list(step_results.values())[-1] if step_results else None
    return final, step_results


# Load Mistral-7B with Outlines
print("Loading Mistral-7B-Instruct with Outlines (structured generation)...")
print("This uses constrained decoding - output is GUARANTEED to match schema")
print("(First load may take a few minutes)")

# Load model through Outlines
llm_gen = outlines.models.transformers(
    "mistralai/Mistral-7B-Instruct-v0.2",
    model_kwargs={
        "load_in_4bit": True,
        "device_map": "auto",
    }
)

# Create JSON generator with Pydantic schema
generator = outlines.generate.json(llm_gen, Decomposition)

print("Mistral-7B + Outlines ready!")
print("Output schema: steps[{left, op, right, result, why}], answer")

# Test on simple problems
test_problems = [
    "John has 5 apples. Mary gives him 3 more. How many apples does John have?",
    "A store has 24 cookies. They split them equally into 4 boxes. How many cookies per box?",
    "Tim has 10 dollars. He buys 3 toys at 2 dollars each. How much does he have left?",
]

print("\n" + "="*60)
print("Testing Outlines Structured Generation")
print("="*60)

for prob in test_problems:
    print(f"\nProblem: {prob}")
    print("-"*40)
    
    decomp = decompose_problem_outlines(prob, generator)
    
    print(f"Steps: {decomp['steps']}")
    print(f"LLM Answer: {decomp['answer']}")
    
    result, history = decomposition_to_execution(decomp)
    print(f"Our execution: {result}")
    print(f"Step history: {history}")

In [ ]:
# LLM Decomposition → Training Integration
# Convert LLM decomposition to prefix_steps format for our training loop

def llm_decomposition_to_prefix_steps(decomposition: dict) -> Tuple[List[str], Dict[str, str]]:
    """
    Convert LLM decomposition to prefix_steps format for training.
    
    Returns: (prefix_steps, num_map)
    """
    steps = decomposition.get('steps', [])
    numbers = decomposition.get('numbers', {})
    
    if not steps:
        return [], {}
    
    # Build num_map from extracted numbers
    num_map = {}
    for var, val in numbers.items():
        # Map a, b, c, ... to NUM_0, NUM_1, ...
        if len(var) == 1 and var.isalpha():
            idx = ord(var.lower()) - ord('a')
            num_map[f"NUM_{idx}"] = str(val)
    
    # Convert steps to prefix notation
    prefix_steps = []
    step_results = {}
    
    for i, step in enumerate(steps):
        op = step['op']
        left = step['left']
        right = step['right']
        result = step['result']
        
        def resolve_operand(operand_str):
            """Resolve operand to NUM_x or step_x reference."""
            # Variable reference (a, b, c, ...)
            if operand_str in numbers:
                idx = ord(operand_str.lower()) - ord('a')
                return f"NUM_{idx}"
            
            # Previous step result
            try:
                val = float(operand_str)
                for prev_name, prev_result in step_results.items():
                    if abs(val - prev_result) < 0.01:
                        return prev_name
                # New constant
                for key, existing_val in num_map.items():
                    if abs(val - float(existing_val)) < 0.01:
                        return key
                const_idx = len([k for k in num_map.keys() if k.startswith('CONST')])
                const_key = f"CONST_{const_idx}"
                num_map[const_key] = str(val)
                return const_key
            except:
                return operand_str
        
        left_ref = resolve_operand(left)
        right_ref = resolve_operand(right)
        
        prefix_steps.append(f"{op} {left_ref} {right_ref}")
        step_results[f"step_{i+1}"] = result
    
    return prefix_steps, num_map


class LLMDecomposedDataset(Dataset):
    """Dataset that decomposes problems using LLM (replaces GTS)."""
    
    def __init__(self, data: List[dict], llm_gen, tokenizer, device, max_samples: int = None):
        self.data = data[:max_samples] if max_samples else data
        self.llm_gen = llm_gen
        self.tokenizer = tokenizer
        self.device = device
        self.cache = {}  # Cache decompositions
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        question = item.get("normalized_question", item.get("question", ""))
        answer = float(item.get("answer", 0))
        
        # Get or compute decomposition
        if idx not in self.cache:
            self.cache[idx] = decompose_problem_llm(question, self.llm_gen, self.tokenizer, self.device)
        decomp = self.cache[idx]
        
        # Convert to prefix format
        prefix_steps, num_map = llm_decomposition_to_prefix_steps(decomp)
        
        # Tokenize
        encoding = self.tokenizer(
            f"solve: {question}",
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "answer": answer,
            "num_map": num_map,
            "question": question,
            "prefix_steps": prefix_steps,
            "num_steps": len(prefix_steps),
        }


# Quick test of the conversion
print("Testing LLM → prefix_steps conversion:")
print("-"*50)

test_decomp = {
    'numbers': {'a': 5.0, 'b': 3.0},
    'steps': [
        {'op': '+', 'left': 'a', 'right': 'b', 'result': 8.0, 'desc': 'add'},
    ],
    'answer': 8.0
}

prefix_steps, num_map = llm_decomposition_to_prefix_steps(test_decomp)
print(f"Input decomp: {test_decomp}")
print(f"Output prefix_steps: {prefix_steps}")
print(f"Output num_map: {num_map}")

# Verify execution
result, _ = execute_multi_step(prefix_steps, num_map)
print(f"Execution result: {result} (expected: 8)")
print("✓ Conversion working!" if result == 8.0 else "✗ Conversion broken!")

In [ ]:
# Test LLM decomposition accuracy with Outlines structured generation
# This should have ZERO parse failures since output is guaranteed valid JSON

print("="*70)
print("OUTLINES STRUCTURED GENERATION ACCURACY TEST")
print("(Zero parse failures expected - constrained decoding)")
print("="*70)

correct = 0
parse_failures = 0
exec_failures = 0
total = 0

NUM_SAMPLES = 30

print(f"\nTesting on {NUM_SAMPLES} problems...")
print("-"*70)

for i, item in enumerate(train_data[:NUM_SAMPLES]):
    question = item.get("normalized_question", item.get("question", ""))
    expected = float(item.get("answer", 0))
    
    # LLM decomposition with Outlines
    decomp = decompose_problem_outlines(question, generator)
    
    # Check if parsing worked (should always work with Outlines)
    if not decomp['steps']:
        parse_failures += 1
        print(f"\n[{i+1}] PARSE FAIL: {question[:50]}...")
        print(f"     Raw: {decomp.get('raw_response', '')[:100]}...")
        total += 1
        continue
    
    # Execute
    result, history = decomposition_to_execution(decomp)
    
    if result is None:
        exec_failures += 1
        print(f"\n[{i+1}] EXEC FAIL: {question[:50]}...")
        print(f"     Steps: {decomp['steps']}")
        total += 1
        continue
    
    # Check answer
    is_correct = abs(result - expected) < 0.01
    
    if is_correct:
        correct += 1
    else:
        print(f"\n[{i+1}] WRONG: {question[:50]}...")
        print(f"     Expected: {expected}, Got: {result}")
        print(f"     LLM answer: {decomp.get('answer')}")
        print(f"     Steps: {decomp['steps'][:3]}...")
    
    total += 1
    
    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/{NUM_SAMPLES}, Accuracy so far: {correct}/{total} = {100*correct/total:.1f}%")

print("\n" + "="*70)
print("OUTLINES STRUCTURED GENERATION RESULTS")
print("="*70)
print(f"Correct:        {correct}/{total} = {100*correct/total:.1f}%")
print(f"Parse failures: {parse_failures} (should be 0 with Outlines)")
print(f"Exec failures:  {exec_failures}")
print(f"Wrong answers:  {total - correct - parse_failures - exec_failures}")
print("="*70)

if correct/total > 0.5:
    print("\n✓ Structured generation working! Ready for training.")
elif correct/total > 0.3:
    print("\n⚠ Moderate accuracy. Model reasoning may need work.")
else:
    print("\n✗ Low accuracy. Model struggling with problem understanding.")

## 5. Training Loop with Tree Growth

In [ ]:
def load_gsm8k_data(path: str) -> List[dict]:
    """Load GSM8K data."""
    with open(path) as f:
        return [json.loads(line) for line in f]


def extract_numbers_from_problem(problem: str) -> Dict[str, float]:
    """Extract numbers from problem text."""
    import re
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', problem)
    return {f"NUM_{i}": float(n) for i, n in enumerate(numbers)}


def prefix_op_to_dsl(op: str) -> str:
    """Map prefix operator to our DSL format."""
    op_map = {'+': 'a + b', '-': 'a - b', '*': 'a * b', '/': 'a / b'}
    return op_map.get(op, 'a + b')


class GSM8KDataset(Dataset):
    def __init__(self, data: List[dict], tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        question = item.get("normalized_question", item.get("question", ""))
        text = f"solve: {question}"
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # Get prefix_steps for multi-step execution
        prefix_steps = item.get("prefix_steps", [])
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "answer": float(item.get("answer", 0)),
            "num_map": item.get("num_map", {}),
            "question": question,
            "prefix_steps": prefix_steps,  # List of atomic operations
            "num_steps": item.get("num_steps", 1),
        }


# Custom collate to handle variable-length prefix_steps
def collate_fn(batch):
    """Custom collate that handles prefix_steps lists."""
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "answer": [b["answer"] for b in batch],
        "num_map": [b["num_map"] for b in batch],
        "question": [b["question"] for b in batch],
        "prefix_steps": [b["prefix_steps"] for b in batch],
        "num_steps": [b["num_steps"] for b in batch],
    }

In [ ]:
from collections import defaultdict

def train_step_two_stream(
    model: TreeBoundLLM,
    tree: SignatureTree,
    batch: dict,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    debug: bool = False,
    confusion: dict = None,
    use_model_routing: bool = False,
) -> Tuple[float, bool, int, int, bool]:
    """
    Two-stream encoding: operation stream for routing, context stream for execution.
    
    Stream 1 (Op): "add", "divide", etc. → routes to signature
    Stream 2 (Ctx): "step 1: 48, 2" → provides values
    
    Returns: (loss, correct_answer, num_steps, correct_routes, model_routed_correct)
    """
    model.train()
    
    # Unpack batch
    answer = batch["answer"][0] if isinstance(batch["answer"], list) else batch["answer"]
    num_map = batch["num_map"][0] if isinstance(batch["num_map"], list) else batch["num_map"]
    prefix_steps = batch["prefix_steps"][0] if isinstance(batch["prefix_steps"], list) else batch["prefix_steps"]
    question = batch["question"][0] if isinstance(batch["question"], list) else batch["question"]
    
    if isinstance(num_map, str):
        num_map = json.loads(num_map)
    
    if not prefix_steps:
        return 0.0, False, 0, 0, False
    
    # Bootstrap: ensure we have all 4 basic operations
    for op, dsl in [('+', 'a + b'), ('-', 'a - b'), ('*', 'a * b'), ('/', 'a / b')]:
        if not tree.has_dsl(dsl):
            tree.add_signature(
                embedding=torch.randn(256).to(device),
                dsl=dsl,
                description=f"bootstrap: {op}"
            )
    
    # TWO-STREAM ENCODING
    op_embeddings, ctx_embeddings = model.encode_two_stream(
        problem_text=question,
        prefix_steps=prefix_steps,
        num_map=num_map,
        device=device
    )
    
    if op_embeddings is None:
        return 0.0, False, 0, 0, False
    
    # Route using ONLY operation embeddings (cleaner signal!)
    queries, route_probs, scores = model.route_with_op_stream(op_embeddings, tree, debug=debug)
    
    if debug:
        print(f"  Two-stream encoding: {len(prefix_steps)} steps")
        print(f"  Op embeddings shape: {op_embeddings.shape}")
        print(f"  Ctx embeddings shape: {ctx_embeddings.shape}")
        print(f"  Route probs shape: {route_probs.shape}")
    
    total_loss = 0.0
    step_results_gt = {}
    step_results_model = {}
    correct_routes = 0
    
    for step_idx, prefix in enumerate(prefix_steps):
        op, left, right = parse_prefix_step(prefix)
        if op is None:
            continue
        
        # Ground truth DSL
        target_dsl = prefix_op_to_dsl(op)
        target_idx = tree.get_id_for_dsl(target_dsl)
        
        # Model's choice (from op_embedding routing)
        step_route_probs = route_probs[step_idx]
        routed_idx = step_route_probs.argmax().item()
        routed_dsl = tree.get_dsl(routed_idx)
        
        # Track confusion matrix
        if confusion is not None:
            confusion[(target_dsl, routed_dsl)] += 1
        
        # Did model route correctly?
        route_correct = (routed_dsl == target_dsl)
        if route_correct:
            correct_routes += 1
        
        if debug:
            op_word = {'+': 'add', '-': 'subtract', '*': 'multiply', '/': 'divide'}.get(op, op)
            print(f"    Step {step_idx+1}: op='{op_word}', target={target_dsl}, routed={routed_dsl}, correct={route_correct}")
            top_k = step_route_probs.topk(min(3, tree.num_active))
            for prob, idx in zip(top_k.values, top_k.indices):
                print(f"      {tree.get_dsl(idx.item())}: {prob.item():.3f}")
        
        # Loss: encourage routing to correct signature
        if target_idx is not None and target_idx < step_route_probs.shape[0]:
            step_loss = -torch.log(step_route_probs[target_idx] + 1e-8)
            total_loss += step_loss
        
        # Ground truth execution
        step_name = f"step_{step_idx + 1}"
        result_gt = execute_prefix_step(prefix, num_map, step_results_gt)
        if result_gt is not None:
            step_results_gt[step_name] = result_gt
        
        # Model-routed execution
        if use_model_routing:
            routed_op = {'+': '+', '-': '-', '*': '*', '/': '/'}.get(
                routed_dsl.replace('a', '').replace('b', '').strip(), op
            )
            routed_prefix = f"{routed_op} {left} {right}"
            result_model = execute_prefix_step(routed_prefix, num_map, step_results_model)
            if result_model is not None:
                step_results_model[step_name] = result_model
    
    # Check final answers
    final_gt = list(step_results_gt.values())[-1] if step_results_gt else None
    correct_gt = final_gt is not None and abs(final_gt - float(answer)) < 0.01
    
    model_routed_correct = False
    if use_model_routing and step_results_model:
        final_model = list(step_results_model.values())[-1]
        model_routed_correct = final_model is not None and abs(final_model - float(answer)) < 0.01
    
    # Backprop
    if isinstance(total_loss, torch.Tensor) and total_loss.requires_grad:
        avg_loss = total_loss / max(len(prefix_steps), 1)
        optimizer.zero_grad()
        avg_loss.backward()
        optimizer.step()
        return avg_loss.item(), correct_gt, len(prefix_steps), correct_routes, model_routed_correct
    
    return 0.0, correct_gt, len(prefix_steps), correct_routes, model_routed_correct


def train_epoch_two_stream(
    model: TreeBoundLLM,
    tree: SignatureTree,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_steps: int = None,
    use_model_routing: bool = False,
) -> dict:
    """Train with two-stream encoding."""
    total_loss = 0
    total_correct = 0
    total_steps = 0
    total_correct_routes = 0
    total_model_correct = 0
    n_problems = 0
    
    confusion = defaultdict(int)
    
    for i, batch in enumerate(dataloader):
        if max_steps and i >= max_steps:
            break
        
        debug = (i == 0)
        
        loss, correct, num_steps, correct_routes, model_correct = train_step_two_stream(
            model, tree, batch, optimizer, device,
            debug=debug,
            confusion=confusion,
            use_model_routing=use_model_routing
        )
        
        total_loss += loss
        total_correct += int(correct)
        total_steps += num_steps
        total_correct_routes += correct_routes
        total_model_correct += int(model_correct)
        n_problems += 1
        
        if (i + 1) % 50 == 0:
            ans_acc = total_correct / n_problems
            route_acc = total_correct_routes / max(total_steps, 1)
            model_acc = total_model_correct / n_problems if use_model_routing else 0
            msg = f"  Step {i+1}: loss={total_loss/n_problems:.4f}, ans_acc={ans_acc:.1%}, route_acc={route_acc:.1%}"
            if use_model_routing:
                msg += f", model_exec={model_acc:.1%}"
            print(msg)
    
    return {
        "loss": total_loss / n_problems,
        "answer_accuracy": total_correct / n_problems,
        "routing_accuracy": total_correct_routes / max(total_steps, 1),
        "model_routed_accuracy": total_model_correct / n_problems if use_model_routing else None,
        "avg_steps": total_steps / n_problems,
        "num_signatures": tree.num_active,
        "confusion": dict(confusion),
    }


def print_confusion_matrix(confusion: dict):
    """Pretty print the confusion matrix."""
    ops = ['a + b', 'a - b', 'a * b', 'a / b']
    
    print("\nConfusion Matrix (target → routed):")
    print("-" * 50)
    
    print(f"{'Target':<10}", end="")
    for op in ops:
        short = op.replace('a ', '').replace(' b', '')
        print(f"{short:>8}", end="")
    print(f"{'Total':>8}")
    
    print("-" * 50)
    
    for target in ops:
        short_t = target.replace('a ', '').replace(' b', '')
        print(f"{short_t:<10}", end="")
        row_total = 0
        for routed in ops:
            count = confusion.get((target, routed), 0)
            row_total += count
            if target == routed:
                print(f"{count:>7}*", end="")
            else:
                print(f"{count:>8}", end="")
        print(f"{row_total:>8}")
    
    print("-" * 50)
    
    print("\nPer-operation accuracy:")
    for target in ops:
        correct = confusion.get((target, target), 0)
        total = sum(confusion.get((target, r), 0) for r in ops)
        if total > 0:
            short = target.replace('a ', '').replace(' b', '')
            print(f"  {short}: {correct}/{total} = {100*correct/total:.1f}%")


# Keep legacy functions
def train_step_per_step_encoding(model, tree, batch, optimizer, device, debug=False, confusion=None, use_model_routing=False):
    """Legacy single-stream encoding."""
    return train_step_two_stream(model, tree, batch, optimizer, device, debug, confusion, use_model_routing)

def train_epoch_per_step(model, tree, dataloader, optimizer, device, max_steps=None, use_model_routing=False):
    """Legacy - now uses two-stream."""
    return train_epoch_two_stream(model, tree, dataloader, optimizer, device, max_steps, use_model_routing)

In [ ]:
# Data already loaded from Hugging Face in cell above
# Just show stats

print(f"Training data: {len(train_data)} problems")
print(f"Test data: {len(test_data)} problems")

# Show a few examples
print("\nSample problems:")
for i, item in enumerate(train_data[:3]):
    print(f"\n[{i+1}] Q: {item['question'][:60]}...")
    print(f"    A: {item['answer']}")

In [ ]:
# Two-stream encoding training
NUM_EPOCHS = 10
STEPS_PER_EPOCH = 300
USE_MODEL_ROUTING = True

history = []

print("="*60)
print("TWO-STREAM ENCODING TRAINING")
print("Stream 1 (Op): 'add', 'divide', etc. → routing")
print("Stream 2 (Ctx): 'step 1: 48, 2' → execution values")
print("-"*60)
print(f"Epochs: {NUM_EPOCHS}, Steps/epoch: {STEPS_PER_EPOCH}")
print(f"Model-routed execution: {USE_MODEL_ROUTING}")
print("="*60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    metrics = train_epoch_two_stream(
        model, tree, train_loader, optimizer, device,
        max_steps=STEPS_PER_EPOCH,
        use_model_routing=USE_MODEL_ROUTING
    )
    
    history.append(metrics)
    
    print(f"\nEpoch {epoch+1} complete:")
    print(f"  Loss: {metrics['loss']:.4f}")
    print(f"  Answer accuracy (GT): {metrics['answer_accuracy']:.1%}")
    print(f"  Routing accuracy: {metrics['routing_accuracy']:.1%}")
    if USE_MODEL_ROUTING:
        print(f"  Model-routed accuracy: {metrics['model_routed_accuracy']:.1%}  <-- END-TO-END")
    
    # Show confusion matrix every 2 epochs
    if (epoch + 1) % 2 == 0 or epoch == NUM_EPOCHS - 1:
        print_confusion_matrix(metrics['confusion'])

print("\n" + "="*60)
print("TWO-STREAM TRAINING COMPLETE")
print(f"Final routing accuracy: {history[-1]['routing_accuracy']:.1%}")
if USE_MODEL_ROUTING:
    print(f"Final model-routed accuracy: {history[-1]['model_routed_accuracy']:.1%}")
print("="*60)

In [ ]:
# Visualize training
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history) + 1)

axes[0].plot(epochs, [h['loss'] for h in history], 'b-o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(epochs, [h['accuracy'] for h in history], 'g-o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')

axes[2].plot(epochs, [h['num_signatures'] for h in history], 'r-o')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('# Signatures')
axes[2].set_title('Tree Growth')

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the tree using summary()
print("="*60)
print("SIGNATURE TREE SUMMARY (Deduplicated by DSL)")
print("="*60)
print(tree.summary())

print("\n" + "="*60)
print("DETAILED SIGNATURES")
print("="*60)
for idx in range(tree.num_active):
    sig = tree.signatures[idx]
    print(f"\n[{idx}] {sig.dsl}")
    print(f"    Description: {sig.description}")
    print(f"    Usage: n={sig.n}, success={sig.mean:.1%}, variance={sig.variance:.3f}")

## 7. Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, tree, test_data, device, max_samples=100):
    """Evaluate on test set."""
    model.eval()
    
    correct = 0
    total = 0
    
    for item in test_data[:max_samples]:
        question = item.get("normalized_question", item.get("question", ""))
        answer = float(item.get("answer", 0))
        num_map = item.get("num_map", {})
        
        # Tokenize
        text = f"solve: {question}"
        inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
        
        # Forward
        query, route_probs, scores, _ = model(
            inputs["input_ids"],
            inputs["attention_mask"],
            tree
        )
        
        # Route
        route_idx = route_probs.argmax(dim=-1).item()
        dsl = tree.get_dsl(route_idx)
        
        # Execute
        operands = {}
        for i, (k, v) in enumerate(num_map.items()):
            if i == 0:
                operands["a"] = float(v)
            elif i == 1:
                operands["b"] = float(v)
        
        result = execute_dsl(dsl, operands, {})
        
        if result is not None and abs(result - answer) < 0.01:
            correct += 1
        
        total += 1
    
    return correct / total if total > 0 else 0


# Run evaluation
test_acc = evaluate(model, tree, test_1step, device)
print(f"\nTest accuracy: {test_acc:.1%}")

In [ ]:
# Save model and tree
import pickle

torch.save({
    "tree_attention": model.tree_attention.state_dict(),
    "base_threshold": model.base_threshold,
}, "tree_bound_llm.pt")

with open("signature_tree.pkl", "wb") as f:
    pickle.dump({
        "signatures": tree.signatures,
        "embeddings": tree.embeddings.cpu(),
        "active_mask": tree.active_mask.cpu(),
        "num_active": tree.num_active,
    }, f)

print("Saved model and tree!")

# Download
files.download("tree_bound_llm.pt")
files.download("signature_tree.pkl")

## 8. Interactive Testing

In [ ]:
@torch.no_grad()
def solve_problem(model, tree, problem, device):
    """Solve a single problem."""
    model.eval()
    
    # Extract numbers
    numbers = extract_numbers_from_problem(problem)
    print(f"Numbers found: {numbers}")
    
    # Tokenize
    text = f"solve: {problem}"
    inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
    
    # Forward
    query, route_probs, scores, should_create = model(
        inputs["input_ids"],
        inputs["attention_mask"],
        tree
    )
    
    # Show routing
    print(f"\nRouting probabilities:")
    for idx in range(tree.num_active):
        prob = route_probs[0, idx].item()
        sig = tree.signatures[idx]
        print(f"  [{idx}] {sig.dsl:10s} : {prob:.1%}")
    
    # Route
    route_idx = route_probs.argmax(dim=-1).item()
    dsl = tree.get_dsl(route_idx)
    print(f"\nSelected: Signature {route_idx} ({dsl})")
    
    # Execute
    operands = {"a": list(numbers.values())[0] if numbers else 0,
                "b": list(numbers.values())[1] if len(numbers) > 1 else 0}
    result = execute_dsl(dsl, operands, {})
    print(f"Result: {result}")
    
    return result


# Try it!
solve_problem(model, tree, "John has 5 apples. Mary gives him 3 more. How many apples?", device)

In [ ]:
# Try more problems
problems = [
    "There are 12 cookies. 4 kids share them equally. How many does each get?",
    "A book costs 15 dollars. You have 8 dollars. How much more do you need?",
    "There are 6 rows with 7 chairs each. How many chairs total?",
]

for p in problems:
    print("\n" + "="*60)
    print(f"Problem: {p}")
    solve_problem(model, tree, p, device)